# JSONBench walkthrough

Generate a small travel dataset in memory.

In [ ]:
from json_bench.data import generate_travel_dataset

dataset = generate_travel_dataset(
    scale=0.1,
    seed=23,
    reference_date="2025-11-01",
)

## Deploy the databases

Start Docker and run `uv sync --locked` first. BONDI waits for each database to be ready.
Couchbase uses Community Edition with a `travel` bucket. Change the names or ports if needed.

In [ ]:
from bondi.deployment import deploy_couchbase, deploy_mongodb

mongo_deployment = deploy_mongodb(name="jsonbench-travel", port=27127)
couchbase_deployment = deploy_couchbase(
    name="jsonbench-travel-cb",
    management_port=18291,
    query_port=18293,
    kv_port=21230,
    bucket="travel",
)
print(mongo_deployment.uri)
print(couchbase_deployment.console_url)

## Load the same dataset into both databases

Load each collection into MongoDB and Couchbase, preserving document IDs and date strings.
BONDI derives Couchbase keys from the IDs and creates no query indexes.
The default `create` mode requires empty collections. Run cleanup before repeating deployment and loading.

In [ ]:
from bondi.couchbase import CouchbaseCollectionRef
from bondi.data import load_documents
from bondi.mongodb import MongoCollectionRef

for name, documents in dataset.items():
    mongo_result = load_documents(
        documents,
        target=mongo_deployment,
        collection=MongoCollectionRef("travel", name),
    )
    couchbase_result = load_documents(
        documents,
        target=couchbase_deployment,
        collection=CouchbaseCollectionRef("travel", "_default", name),
    )
    print(mongo_result.destinations, couchbase_result.destinations)

## Build Q1–Q8 workloads

Translate all eight templates into one ordered BONDI workload per database.

In [ ]:
from pathlib import Path
from pprint import pprint

from bondi.workload import Workload, WorkloadEntry

from json_bench.queries import load_template, translate_couchbase, translate_mongodb

# Support kernels started in either the repository root or notebooks/.
project_root = Path.cwd()
if not (project_root / "experiments" / "templates").is_dir():
    project_root = project_root.parent

template_dir = project_root / "experiments" / "templates"
templates = [load_template(template_dir / f"q{i}.json") for i in range(1, 9)]

# Change Q4's parameter in memory; keep the saved paper template unchanged.
q4 = templates[3]
q4["operations"]["match"]["itinerary.location"] = dataset["tours"][0]["itinerary"][0]["location"]
print("Q4 location:", q4["operations"]["match"]["itinerary.location"])

mongo_workload = Workload(
    [WorkloadEntry(translate_mongodb(t, database="travel")) for t in templates]
)
couchbase_workload = Workload(
    [WorkloadEntry(translate_couchbase(t, bucket="travel")) for t in templates]
)

for template, mongo_entry, couchbase_entry in zip(
    templates, mongo_workload, couchbase_workload, strict=True
):
    print(template["name"])
    pprint(mongo_entry.query.pipeline)
    print(couchbase_entry.query.statement)

## Execute the workloads

Run each query once and retain its documents. Create the original bookings, tours and reviews indexes first.
Use REQUEST_PLUS to include newly loaded data in Couchbase. This is a correctness check, not a benchmark.

In [ ]:
from bondi.couchbase import CouchbaseExecutor, CouchbaseQuery
from bondi.mongodb import MongoExecutor
from couchbase.n1ql import QueryScanConsistency

with MongoExecutor.from_uri(mongo_deployment.uri, timeout_seconds=60) as executor:
    mongo_results = executor.run_workload(mongo_workload, collect_results=True)

with CouchbaseExecutor.from_uri(
    couchbase_deployment.uri,
    username=couchbase_deployment.username,
    password=couchbase_deployment.password,
    timeout_seconds=60,
) as executor:
    for statement in [
        "CREATE INDEX `reviews-tour_id` IF NOT EXISTS ON `reviews`(`tour_id`);",
        "CREATE INDEX `bookings-tour_id` IF NOT EXISTS ON `bookings`(`tour_id`);",
        "CREATE INDEX `tours-id` IF NOT EXISTS ON `tours`(`_id`);",
    ]:
        executor.run(
            CouchbaseQuery(statement, query_context=couchbase_workload[0].query.query_context)
        )
    couchbase_results = executor.run_workload(
        couchbase_workload,
        collect_results=True,
        query_kwargs={"scan_consistency": QueryScanConsistency.REQUEST_PLUS},
    )

## Compare query results

Compare each query in workload order. Require the same tour groups and aggregate fields,
ignore row order, and allow small floating-point differences.

In [ ]:
from json_bench.validate import compare_results

for template, mongo_result, couchbase_result in zip(
    templates, mongo_results, couchbase_results, strict=True
):
    compare_results(template, mongo_result, couchbase_result)
    print(f"{template['name']}: results match")

## Generate sustained load

Run MongoDB, then Couchbase, with only one database running at a time.
Use equal explicit weights for Q1–Q8. Each client selects its next query after the previous one finishes.
This short example demonstrates execution; it is not the paper's measurement protocol.

Retain every `LoadResult`, including warm-up, failures and drain expiry. Set `collect_results=False`
to avoid retaining database documents. The same seed reproduces each client's selection prefix,
not identical attempt counts across databases. Both deployments remain stopped after this cell.


In [ ]:
from bondi.load import WeightedRandom

QUERY_WEIGHTS = [1, 1, 1, 1, 1, 1, 1, 1]  # Q1–Q8; edit to change the mix.
LOAD_SEED = 23
CLIENTS = 2
WARMUP_SECONDS = 5
MEASUREMENT_SECONDS = 15
QUERY_TIMEOUT_SECONDS = 10
DRAIN_TIMEOUT_SECONDS = 15

assert len(QUERY_WEIGHTS) == len(templates)
load_results = {}  # Backend name -> list of BONDI LoadResult objects.

In [ ]:
# Stop both first, including when this cell is run again.
mongo_deployment.stop()
couchbase_deployment.stop()

for backend, deployment, workload, executor_type, credentials in [
    ("mongodb", mongo_deployment, mongo_workload, MongoExecutor, {}),
    (
        "couchbase",
        couchbase_deployment,
        couchbase_workload,
        CouchbaseExecutor,
        {"username": couchbase_deployment.username, "password": couchbase_deployment.password},
    ),
]:
    print(f"Running {backend} with {CLIENTS} clients...", flush=True)
    try:
        deployment.start()
        with executor_type.from_uri(
            deployment.uri,
            **credentials,
            concurrency=CLIENTS,
            timeout_seconds=QUERY_TIMEOUT_SECONDS,
            continue_on_error=True,
        ) as executor:
            load_results[backend] = executor.generate_load(
                workload,
                selection=WeightedRandom(weights=QUERY_WEIGHTS, seed=LOAD_SEED),
                warmup_seconds=WARMUP_SECONDS,
                measurement_seconds=MEASUREMENT_SECONDS,
                drain_timeout_seconds=DRAIN_TIMEOUT_SECONDS,
                collect_results=False,
                return_load_results=True,
            )
        print(f"Retained {len(load_results[backend])} observations.")
    finally:
        deployment.stop()

## Inspect retained observations

Show per-query outcomes for measurement admissions, including those that finish during draining.
These counts are not throughput. Raw objects remain in `load_results`; the table keeps phase,
timing, client and stop-reason fields for later analysis. `duration` includes health preflight.


In [ ]:
import pandas as pd
from IPython.display import display

load_observations = pd.DataFrame(
    [
        {
            "backend": backend,
            "query": templates[item.workload_position]["name"],
            "attempt_id": item.attempt_id,
            "client_id": item.client_id,
            "phase": item.phase_started,
            "admitted_seconds": item.admitted_seconds,
            "ended_seconds": item.ended_seconds,
            "duration_seconds": item.duration,
            "outcome": item.result.status if item.result is not None else "unresolved",
            "stop_reason": item.stop_reason,
        }
        for backend, observations in load_results.items()
        for item in observations
    ]
)
measurement_observations = load_observations.loc[load_observations["phase"] == "measurement"]
display(
    measurement_observations.groupby(["backend", "query", "outcome"]).size().unstack(fill_value=0)
)
display(load_observations.head())

## Clean up when finished

Remove both deployments, including their data and saved configuration.

In [ ]:
mongo_deployment.remove()
couchbase_deployment.remove()